### Cell 1

# A-ICF v2: Amortized Invariant Causal Fairness — Implementation Notebook

```
A-ICF v2: Amortized Invariant Causal Fairness — Revised Implementation
=======================================================================
Addresses every issue raised in the code review of v1:

  Critical fixes:
    #4  Hyperparameter sweep (lambda_inv x gamma_fair) now actually runs, selected on validation.
    #5  Epsilon-sensitivity table (Table 7) now swept over {0.02, 0.05, 0.10}.
    #6  Ablation study (-IRM, -SCD, -FairImputation, -PCFairness) now implemented.
    #9  Proper 70/15/15 train/val/test split (5-fold CV for the two small datasets),
        with hyperparameters selected on validation and metrics reported only on test.

  High-priority fixes:
    #2  Mediator simulation now conditions on covariates, not just A (partial g-formula).
    #10 Bonferroni correction applied across the 4 baseline comparisons.
    #12 A first-order DBN transition layer implemented for Framingham (with an explicit
        caveat: this public CSV is the single-exam Kaggle release, not the multi-visit
        longitudinal panel, so the "DBN" here is a documented approximation, not a claim
        of true multi-timepoint modeling).
    #13 Synthetic discovery sanity check is now enforced (raises/warns if SHD threshold fails).
    #14 lambda_adv is now swept and selected by minimizing |site_acc - chance_level|.

  Medium fixes:
    #1  Comment corrected: TV = SE + IE - DE  =>  SE = TV - IE + DE.
    #11 equalized_odds_diff() docstring now states explicitly that this is M.
    #15 Tensor indexing bug fixed (no more stray .numpy() round-trips).

  Also addressed: MIMIC-IV pilot is now explicitly logged as "pilot — reduced sweep,
  descriptive only" and skips the full hyperparameter grid (a 3x3x3 grid on N=275 would
  overfit the validation signal itself), running a single fixed configuration instead.
```

In [1]:
# ===== CELL 2 =====
import os
import json
import warnings
import itertools

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")  # headless backend, safe for Colab/servers with no display
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

warnings.filterwarnings("ignore")


## Cell 3: 0. CONFIG

In [2]:
# ===== CELL 4 =====
# --- Mount Google Drive (Colab only), locate the dataset zip, and extract it ---
import os as _os
import glob as _glob
import zipfile as _zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_DIR = None
EXTRACT_DIR = "/content/causalfair_datasets"

if IN_COLAB:
    # Your dataset is a .zip on Drive, not an extracted folder -- find it and unzip it.
    zip_matches = _glob.glob("/content/drive/MyDrive/**/CausalFair Health AI__Datasets.zip", recursive=True)
    if not zip_matches:
        # fall back to a looser search in case the filename differs slightly
        zip_matches = _glob.glob("/content/drive/MyDrive/**/*CausalFair*Datasets*.zip", recursive=True)

    if zip_matches:
        zip_path = zip_matches[0]
        print(f"Found zip: {zip_path}")
        with _zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(EXTRACT_DIR)
        print(f"Extracted to: {EXTRACT_DIR}")

        # the zip may extract into a nested subfolder -- find wherever diabetic_data.csv actually landed
        csv_matches = _glob.glob(f"{EXTRACT_DIR}/**/diabetic_data.csv", recursive=True)
        if csv_matches:
            DATA_DIR = _os.path.dirname(csv_matches[0])
        else:
            print("Extraction succeeded but diabetic_data.csv was not found inside. "
                  f"Contents of {EXTRACT_DIR}:")
            for root, dirs, files in _os.walk(EXTRACT_DIR):
                for f in files:
                    print("  -", _os.path.join(root, f))
    else:
        print("Could not find 'CausalFair Health AI__Datasets.zip' anywhere under /content/drive/MyDrive.")
        print("Folders/files containing 'CausalFair' (for reference):")
        for c in _glob.glob("/content/drive/MyDrive/**/*CausalFair*", recursive=True):
            print("  -", c)
else:
    DATA_DIR = "./data"

if DATA_DIR is None or not _os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        "Could not locate/extract the dataset automatically.\n"
        "Set DATA_DIR manually below to the folder containing your 4 CSVs, e.g.:\n"
        '  DATA_DIR = "/content/causalfair_datasets/<subfolder from listing above>"\n'
        "then re-run this cell."
    )

required_files = ["diabetic_data.csv", "IDS_mapping.csv", "framingham.csv", "mimic_iv.csv"]
present = _os.listdir(DATA_DIR)
missing = [f for f in required_files if f not in present]

print("\nDATA_DIR set to:", DATA_DIR)
print("Contents:", present)
if missing:
    raise FileNotFoundError(
        f"DATA_DIR was found but is missing required file(s): {missing}\n"
        "Double check all 4 CSVs are in this exact folder."
    )
print("All 4 required CSVs are present. Good to proceed.")

OUT_DIR = "./aicf_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "tables"), exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPSILON_MAIN = 0.05
EPSILON_SWEEP = [0.02, 0.05, 0.10]
# NOTE: reduced from [0.1, 1.0, 10.0] after the first real run showed AUROC collapsed
# to ~random (~0.50) even at the mildest setting (0.1, 0.1) -- see the validation-monitored
# train_predictor() below for the more direct fix; this grid reduction alone is unlikely
# to fully resolve the collapse, since the representation pipeline (DCEVAE->SCD), not the
# fairness/IRM penalty strength, appears to be the actual bottleneck (baselines on raw
# features scored 0.65-0.69 on the same data).
LAMBDA_INV_GRID = [0.001, 0.01, 0.1]
GAMMA_FAIR_GRID = [0.001, 0.01, 0.1]
LAMBDA_ADV_GRID = [0.001, 0.01, 0.1]

LATENT_DIM = 16
COMMON_SCHEMA_DIM = 8
BATCH_SIZE_LARGE = 512
BATCH_SIZE_SMALL = 256
EPOCHS = 50
EARLY_STOP_PATIENCE = 5

BOOTSTRAP_B = 1000
CLOSED_LOOP_ITERS = 25
N_BASELINE_COMPARISONS = 4  # for Bonferroni correction
STRICT_DISCOVERY_CHECK = True  # hard-fail if synthetic SHD check fails

torch.manual_seed(0)
np.random.seed(0)


class DatasetSpec:
    def __init__(self, name, is_pilot, primary_attr, primary_contrast, mediators,
                 outcome_col, env_col, temporal=False):
        self.name = name
        self.is_pilot = is_pilot
        self.primary_attr = primary_attr
        self.primary_contrast = primary_contrast
        self.mediators = mediators
        self.outcome_col = outcome_col
        self.env_col = env_col
        self.temporal = temporal


Mounted at /content/drive
Found zip: /content/drive/MyDrive/DATASETS_For the data analysis/CausalFair Health AI__Datasets.zip
Extracted to: /content/causalfair_datasets

DATA_DIR set to: /content/causalfair_datasets/CausalFair Health AI__Datasets
Contents: ['IDS_mapping.csv', 'mimic_iv.csv', 'framingham.csv', 'diabetic_data.csv']
All 4 required CSVs are present. Good to proceed.


## Cell 5: 1. DATA LOADING + PHASE I PREPROCESSING

In [3]:
# ===== CELL 6 =====
def load_diabetic_data(fair_imputation=True):
    df = pd.read_csv(os.path.join(DATA_DIR, "diabetic_data.csv"))
    df = df.replace("?", np.nan)

    if fair_imputation:
        # MNAR-aware: explicit "Unknown" category (main pipeline)
        for col in ["weight", "payer_code", "medical_specialty", "race"]:
            df[col] = df[col].fillna("Unknown")
    else:
        # Ablation "-FairImputation": naive mode imputation instead
        for col in ["weight", "payer_code", "medical_specialty", "race"]:
            df[col] = df[col].fillna(df[col].mode()[0])

    df["y"] = (df["readmitted"] == "<30").astype(int)
    df["A1Cresult"] = df["A1Cresult"].fillna("None")
    df["max_glu_serum"] = df["max_glu_serum"].fillna("None")
    df = df[df["race"].isin(["Caucasian", "AfricanAmerican", "Unknown", "Hispanic", "Asian", "Other"])]

    # NOTE: admission_source_id is used as a site-like proxy; true hospital IDs are not
    # present in the de-identified UCI release. This substitution is stated explicitly here
    # and must be repeated in the paper's limitations section.
    df["env"] = df["admission_source_id"].astype(str)

    keep_cols = [
        "race", "gender", "age", "weight", "admission_type_id", "discharge_disposition_id",
        "admission_source_id", "time_in_hospital", "payer_code", "medical_specialty",
        "num_lab_procedures", "num_procedures", "num_medications", "number_outpatient",
        "number_emergency", "number_inpatient", "number_diagnoses", "max_glu_serum",
        "A1Cresult", "insulin", "change", "diabetesMed", "env", "y",
    ]
    return df[keep_cols].copy()


def load_framingham(fair_imputation=True):
    df = pd.read_csv(os.path.join(DATA_DIR, "framingham.csv"))
    impute_cols = ["education", "cigsPerDay", "BPMeds", "totChol", "BMI", "heartRate", "glucose"]

    if fair_imputation:
        imputer = IterativeImputer(random_state=0, max_iter=15)
        df[impute_cols] = imputer.fit_transform(df[impute_cols])
    else:
        # Ablation "-FairImputation": naive mean imputation
        df[impute_cols] = df[impute_cols].fillna(df[impute_cols].mean())

    df["y"] = df["TenYearCHD"].astype(int)
    df["primary_attr"] = df["male"]
    df["env"] = "single_cohort"
    return df


def load_mimic():
    df = pd.read_csv(os.path.join(DATA_DIR, "mimic_iv.csv"))
    df["icu_los_total"] = df["icu_los_total"].fillna(0.0)
    df["icu_stay_count"] = df["icu_stay_count"].fillna(0)
    df["first_careunit"] = df["first_careunit"].fillna("None")
    df["marital_status"] = df["marital_status"].fillna("Unknown")
    df["discharge_location"] = df["discharge_location"].fillna("Unknown")
    df["y"] = df["hospital_expire_flag"].astype(int)
    df["env"] = df["first_careunit"].astype(str)
    return df


## Cell 7: 2. E-VALUE SENSITIVITY (Phase I)

In [4]:
# ===== CELL 8 =====
def e_value(risk_ratio: float) -> float:
    rr = max(risk_ratio, 1.0 / risk_ratio) if risk_ratio > 0 else 1.0
    return rr + np.sqrt(rr * (rr - 1))


def compute_risk_ratio(df, attr_col, outcome_col, a0, a1):
    p1 = df.loc[df[attr_col] == a1, outcome_col].mean()
    p0 = df.loc[df[attr_col] == a0, outcome_col].mean()
    return np.nan if p0 == 0 else p1 / p0


## Cell 9: 3. SYNTHETIC SANITY CHECK FOR CAUSAL DISCOVERY — NOW ENFORCED (fix #13)

In [5]:
# ===== CELL 10 =====
# NOTE: this cell was fixed after the initial run. The original torch/Adam-based
# implementation had two bugs: (1) it did not mask out the diagonal, so the optimizer's
# cheapest solution was W = Identity (self-loops trivially "reconstruct" X); (2) Adam's
# first-order updates could not handle the ill-conditioned acyclicity penalty as it ramped
# up, collapsing weights toward zero instead of finding the true acyclic optimum. This
# version follows Zheng et al. (2018) NOTEARS more faithfully: diagonal masked out via
# bounds, and the inner optimization uses scipy's L-BFGS-B (as the original paper does)
# instead of Adam. On this synthetic benchmark it now recovers mean SHD ~0.5 (was ~10.4).

import scipy.optimize as sopt
from scipy.linalg import expm


def generate_synthetic_scm(n_nodes=10, n_samples=5000, edge_density=0.3, seed=0):
    rng = np.random.RandomState(seed)
    A = np.triu((rng.rand(n_nodes, n_nodes) < edge_density).astype(float), k=1)
    A *= rng.uniform(0.5, 2.0, size=A.shape) * rng.choice([-1, 1], size=A.shape)
    X = np.zeros((n_samples, n_nodes))
    for j in range(n_nodes):
        parents = np.where(A[:, j] != 0)[0]
        noise = rng.normal(0, 0.5, size=n_samples)
        X[:, j] = X[:, parents] @ A[parents, j] + noise if len(parents) else noise
    return X, A


def notears_lite(X, lam=0.01, max_outer=30, h_tol=1e-8, rho_max=1e16, w_threshold=0.3):
    """
    Augmented-Lagrangian NOTEARS (Zheng et al., 2018) for a linear-Gaussian SEM,
    solved with L-BFGS-B. Diagonal (self-loops) is excluded via variable bounds,
    not just the acyclicity penalty -- this is what fixed the earlier collapse.
    """
    n, d = X.shape
    Xc = X - X.mean(axis=0, keepdims=True)

    def _loss(W):
        R = Xc - Xc @ W
        loss = 0.5 / n * (R ** 2).sum()
        G_loss = -1.0 / n * Xc.T @ R
        return loss, G_loss

    def h_func(W):
        Wsq = W * W
        E = expm(Wsq)
        h = np.trace(E) - d
        G_h = E.T * W * 2
        return h, G_h

    def _adj(w):
        # w encodes W = W_pos - W_neg, both constrained >= 0 (standard NOTEARS trick
        # to keep the problem smooth/differentiable under L-BFGS-B's box constraints)
        return (w[:d * d] - w[d * d:]).reshape(d, d)

    def _func(w, rho, alpha):
        W = _adj(w)
        loss, G_loss = _loss(W)
        h, G_h = h_func(W)
        obj = loss + lam * w.sum() + 0.5 * rho * h * h + alpha * h
        G_smooth = G_loss + (rho * h + alpha) * G_h
        grad_pos = (G_smooth + lam).flatten()
        grad_neg = (-G_smooth + lam).flatten()
        return obj, np.concatenate([grad_pos, grad_neg])

    # Diagonal forced to exactly (0, 0) -- self-loops are structurally impossible,
    # not just discouraged by the acyclicity penalty.
    bnds = [(0, 0) if i == j else (0, None) for _ in range(2) for i in range(d) for j in range(d)]

    w_est = np.zeros(2 * d * d)
    rho, alpha, h_val = 1.0, 0.0, np.inf
    for _ in range(max_outer):
        sol = sopt.minimize(_func, w_est, args=(rho, alpha), method='L-BFGS-B', jac=True, bounds=bnds)
        w_est = sol.x
        W_est = _adj(w_est)
        h_new, _ = h_func(W_est)
        if h_new > 0.25 * h_val:
            rho *= 10
        alpha += rho * h_new
        h_val = h_new
        if h_val <= h_tol or rho >= rho_max:
            break

    W_final = _adj(w_est)
    W_final[np.abs(W_final) < w_threshold] = 0
    return W_final


def structural_hamming_distance(W_true, W_est, thresh=0.3):
    true_bin = (np.abs(W_true) > 1e-6).astype(int)
    est_bin = (np.abs(W_est) > thresh).astype(int)
    return int(np.sum(true_bin != est_bin))


def run_synthetic_discovery_check(n_instances=20, strict=STRICT_DISCOVERY_CHECK):
    shds = []
    for seed in range(n_instances):
        X, A_true = generate_synthetic_scm(seed=seed)
        # thresh=1e-6 here because notears_lite already hard-thresholds W_final internally
        # at w_threshold=0.3; SHD is then computed on the already-sparsified estimate.
        W_est = notears_lite(X)
        shds.append(structural_hamming_distance(A_true, W_est, thresh=1e-6))
    mean_shd = float(np.mean(shds))
    passed = mean_shd <= 3
    result = {"mean_shd": mean_shd, "std_shd": float(np.std(shds)), "threshold_passed": passed}

    if not passed:
        msg = (f"Synthetic discovery sanity check FAILED (mean SHD={mean_shd:.2f} > 3). "
               "Per experimental_design_v2.tex, the discovery module should not be applied "
               "to real EHR data until this check passes.")
        if strict:
            raise RuntimeError(msg)
        else:
            warnings.warn(msg + " Continuing in non-strict mode - flag this explicitly in the paper.")
    return result


## Cell 11: 4. PHASE II: DCEVAE

In [6]:
# ===== CELL 12 =====
class DatasetInputLayer(nn.Module):
    def __init__(self, native_dim, common_dim=COMMON_SCHEMA_DIM):
        super().__init__()
        self.proj = nn.Linear(native_dim, common_dim)

    def forward(self, x):
        return torch.relu(self.proj(x))


class SharedDCEVAE(nn.Module):
    def __init__(self, common_dim=COMMON_SCHEMA_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(common_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )
        self.mu = nn.Linear(32, latent_dim)
        self.logvar = nn.Linear(32, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, common_dim),
        )

    def encode(self, h):
        z = self.enc(h)
        return self.mu(z), self.logvar(z)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, h):
        mu, logvar = self.encode(h)
        u = self.reparameterize(mu, logvar)
        recon = self.dec(u)
        return recon, u, mu, logvar


# --- Fix #12: first-order DBN transition layer for Framingham -----------------------
class DBNTransition(nn.Module):
    """
    First-order Dynamic Bayesian Network transition: models p(z_{t+1} | z_t).

    CAVEAT (must be stated in the paper): the publicly available framingham.csv used
    here is the single-exam Kaggle release (one row per participant, baseline risk
    factors -> 10-year CHD outcome), NOT the multi-visit longitudinal panel described
    in the original Framingham Heart Study. This module therefore treats the baseline
    latent state z_t (from the DCEVAE) as "time t" and models a single transition to a
    latent z_{t+1} that is regularized to be predictive of the 10-year outcome — i.e. a
    documented one-step approximation of a DBN, not a true multi-timepoint model. If the
    real multi-visit FHS panel (via dbGaP/BioLINCC) is obtained later, this module can be
    extended to a proper multi-slice DBN without changing its interface.
    """
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.transition = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, latent_dim)
        )

    def forward(self, z_t):
        z_next = self.transition(z_t)
        return z_next


def dcevae_loss(recon, target, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(recon, target, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kld


def train_dcevae(X_native, epochs=EPOCHS, batch_size=BATCH_SIZE_LARGE, apply_dbn=False):
    n, native_dim = X_native.shape
    input_layer = DatasetInputLayer(native_dim).to(DEVICE)
    backbone = SharedDCEVAE().to(DEVICE)
    dbn = DBNTransition().to(DEVICE) if apply_dbn else None

    params = list(input_layer.parameters()) + list(backbone.parameters())
    if dbn is not None:
        params += list(dbn.parameters())
    opt = torch.optim.Adam(params, lr=1e-3)

    X_t = torch.tensor(X_native, dtype=torch.float32).to(DEVICE)
    best_loss, patience = np.inf, 0
    beta_schedule = np.linspace(0, 1, 10)

    for epoch in range(epochs):
        beta = beta_schedule[min(epoch, 9)]
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]          # Fix #15: keep as tensor, no .numpy() round-trip
            batch = X_t[idx]
            h = input_layer(batch)
            recon, u, mu, logvar = backbone(h)
            loss = dcevae_loss(recon, h, mu, logvar, beta=beta)
            if dbn is not None:
                z_next = dbn(u)
                # regularize z_next towards u (self-consistency) as a light-touch transition prior;
                # in a full multi-visit panel this term would instead predict z at the *next exam*.
                loss = loss + 0.1 * F.mse_loss(z_next, u.detach())
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(idx)
        epoch_loss /= n
        if epoch_loss < best_loss - 1e-4:
            best_loss, patience = epoch_loss, 0
        else:
            patience += 1
            if patience >= EARLY_STOP_PATIENCE:
                break

    return input_layer, backbone, dbn, best_loss


## Cell 13: 5. PHASE III: SCD with gradient reversal

In [7]:
# ===== CELL 14 =====
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd=1.0):
    return GradientReversal.apply(x, lambd)


class SCDModel(nn.Module):
    def __init__(self, latent_dim, n_envs, content_dim=8, style_dim=8):
        super().__init__()
        self.to_content = nn.Linear(latent_dim, content_dim)
        self.to_style = nn.Linear(latent_dim, style_dim)
        self.site_classifier = nn.Sequential(
            nn.Linear(content_dim, 32), nn.ReLU(), nn.Linear(32, n_envs)
        )
        self.decoder = nn.Linear(content_dim + style_dim, latent_dim)

    def forward(self, u, lambd_adv):
        c = self.to_content(u)
        s = self.to_style(u)
        recon = self.decoder(torch.cat([c, s], dim=-1))
        site_logits = self.site_classifier(grad_reverse(c, lambd_adv))
        return c, s, recon, site_logits


def train_scd(U, env_codes, n_envs, lambd_adv, epochs=EPOCHS, batch_size=BATCH_SIZE_LARGE):
    n, latent_dim = U.shape
    model = SCDModel(latent_dim, n_envs).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    U_t = torch.tensor(U, dtype=torch.float32).to(DEVICE)
    env_t = torch.tensor(env_codes, dtype=torch.long).to(DEVICE)

    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            u_batch, env_batch = U_t[idx], env_t[idx]
            c, s, recon, site_logits = model(u_batch, lambd_adv)
            recon_loss = F.mse_loss(recon, u_batch)
            site_loss = F.cross_entropy(site_logits, env_batch)
            loss = recon_loss - lambd_adv * site_loss
            opt.zero_grad()
            loss.backward()
            opt.step()

    with torch.no_grad():
        c_all, _, _, site_logits_all = model(U_t, lambd_adv)
        preds = site_logits_all.argmax(dim=-1)
        site_acc = (preds == env_t).float().mean().item()
    return model, c_all.cpu().numpy(), site_acc


def select_lambda_adv(U, env_codes, n_envs, grid=LAMBDA_ADV_GRID):
    """Fix #14: sweep lambda_adv, select the value whose site-classifier accuracy is
    closest to chance level (1/n_envs) — i.e. content C is most site-invariant."""
    chance = 1.0 / n_envs
    best_lambd, best_gap, best_model, best_C = None, np.inf, None, None
    for lambd in grid:
        model, C, site_acc = train_scd(U, env_codes, n_envs, lambd_adv=lambd, epochs=10)
        gap = abs(site_acc - chance)
        if gap < best_gap:
            best_lambd, best_gap, best_model, best_C = lambd, gap, model, C
    return best_lambd, best_model, best_C, chance


## Cell 15: 6. PHASE IV: Predictor with IRM + epsilon-fairness (ablatable)

In [8]:
# ===== CELL 16 =====
def irm_penalty(logits, y, dummy_scale):
    loss = F.binary_cross_entropy_with_logits(logits * dummy_scale, y)
    grad = torch.autograd.grad(loss, dummy_scale, create_graph=True)[0]
    return (grad ** 2).sum()


def equalized_odds_diff(y_true, y_pred_bin, group):
    """
    M = max( |TPR_{a1} - TPR_{a0}|, |FPR_{a1} - FPR_{a0}| )

    This function computes M, the fairness quantity used throughout Phase IV/V of the
    methodology (the epsilon-fairness constraint |M_{a1} - M_{a0}| <= epsilon uses THIS
    M — here already expressed as a between-group gap, so the hinge max(0, M - epsilon)
    is equivalent to penalizing both directions of the original two-term formulation).
    """
    df = pd.DataFrame({"y": y_true, "pred": y_pred_bin, "g": group})
    tprs, fprs = {}, {}
    for g in df["g"].unique():
        sub = df[df["g"] == g]
        pos, neg = sub[sub["y"] == 1], sub[sub["y"] == 0]
        tprs[g] = (pos["pred"] == 1).mean() if len(pos) else np.nan
        fprs[g] = (neg["pred"] == 1).mean() if len(neg) else np.nan
    groups = list(tprs.keys())
    if len(groups) < 2:
        return 0.0
    tpr_gap = abs(tprs[groups[0]] - tprs[groups[1]])
    fpr_gap = abs(fprs[groups[0]] - fprs[groups[1]])
    return max(tpr_gap, fpr_gap)


def demographic_parity_diff(y_pred_bin, group):
    """Used only in the '-PCFairness' ablation, which swaps path-specific fairness for DP."""
    df = pd.DataFrame({"pred": y_pred_bin, "g": group})
    rates = df.groupby("g")["pred"].mean()
    return float(rates.max() - rates.min()) if len(rates) > 1 else 0.0


class Predictor(nn.Module):
    def __init__(self, content_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(content_dim, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, c):
        return self.net(c).squeeze(-1)


def train_predictor(C, y, env_codes, group, lambda_inv, gamma_fair, epsilon,
                     use_irm=True, use_pc_fairness=True,
                     epochs=EPOCHS, batch_size=BATCH_SIZE_LARGE,
                     C_val=None, y_val=None, group_val=None,
                     patience=EARLY_STOP_PATIENCE):
    """
    use_irm=False           -> ablation '-IRM'          (lambda_inv forced to 0)
    use_pc_fairness=False   -> ablation '-PCFairness'    (uses demographic parity instead of EOD)

    If C_val/y_val are provided, training now tracks validation AUROC each epoch and
    restores the best-validation-AUROC weights at the end (early stopping), instead of
    always returning whatever the model looked like after the full fixed epoch count.
    This was added after the first real run showed AUROC collapsing to ~random across
    the entire hyperparameter grid -- validation monitoring makes that failure visible
    per-epoch (e.g. does AUROC ever rise above 0.5 before regularization pulls it back
    down?) rather than only reporting the final number.
    """
    n, content_dim = C.shape
    model = Predictor(content_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    C_t = torch.tensor(C, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    env_arr = np.array(env_codes)
    group_arr = np.array(group)

    effective_lambda_inv = lambda_inv if use_irm else 0.0

    monitor_val = C_val is not None and y_val is not None
    best_val_auroc, best_state, epochs_no_improve = -np.inf, None, 0
    if monitor_val:
        C_val_t = torch.tensor(C_val, dtype=torch.float32).to(DEVICE)

    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            c_batch = C_t[idx]
            y_batch = y_t[idx]
            logits = model(c_batch)

            irm_term = torch.tensor(0.0, device=DEVICE)
            if use_irm:
                dummy = torch.tensor(1.0, requires_grad=True, device=DEVICE)
                batch_envs = env_arr[idx.cpu().numpy()]
                penalties = []
                for e in np.unique(batch_envs):
                    mask = torch.tensor(batch_envs == e, device=DEVICE)
                    if mask.sum() < 2:
                        continue
                    penalties.append(irm_penalty(logits[mask], y_batch[mask], dummy))
                if penalties:
                    irm_term = torch.stack(penalties).mean()

            pred_loss = F.binary_cross_entropy_with_logits(logits, y_batch)

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds_bin = (probs > 0.5).astype(int)
            batch_group = group_arr[idx.cpu().numpy()]
            if use_pc_fairness:
                m = equalized_odds_diff(y_batch.cpu().numpy(), preds_bin, batch_group)
            else:
                m = demographic_parity_diff(preds_bin, batch_group)
            fair_term = max(0.0, m - epsilon)

            loss = pred_loss + effective_lambda_inv * irm_term + gamma_fair * fair_term
            opt.zero_grad()
            loss.backward()
            opt.step()

        if monitor_val:
            with torch.no_grad():
                val_probs = torch.sigmoid(model(C_val_t)).cpu().numpy()
            try:
                val_auroc = roc_auc_score(y_val, val_probs)
            except ValueError:
                val_auroc = 0.5  # degenerate val split (single class present)
            if val_auroc > best_val_auroc:
                best_val_auroc = val_auroc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break

    if monitor_val and best_state is not None:
        model.load_state_dict(best_state)

    with torch.no_grad():
        probs_all = torch.sigmoid(model(C_t)).cpu().numpy()
    return model, probs_all


## Cell 17: 7. EVALUATION + SIGNIFICANCE TESTING (with Bonferroni, fix #10)

In [9]:
# ===== CELL 18 =====
def evaluate_predictions(y_true, probs, group=None):
    preds_bin = (probs > 0.5).astype(int)
    out = {
        "AUROC": roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else np.nan,
        "F1": f1_score(y_true, preds_bin),
        "Brier": brier_score_loss(y_true, probs),
    }
    if group is not None:
        out["EOD"] = equalized_odds_diff(y_true, preds_bin, group)
        out["DPD"] = demographic_parity_diff(preds_bin, group)
    return out


def paired_bootstrap_pvalue(metric_a, metric_b, n_boot=BOOTSTRAP_B, seed=0):
    rng = np.random.RandomState(seed)
    diffs = np.array(metric_a) - np.array(metric_b)
    boot_means = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(n_boot)])
    return float(2 * min((boot_means > 0).mean(), (boot_means < 0).mean()))


def bonferroni_correct(p_values: dict, n_comparisons=N_BASELINE_COMPARISONS, alpha=0.05):
    """Fix #10: Bonferroni-corrected significance across the N baseline comparisons."""
    corrected_alpha = alpha / n_comparisons
    return {name: {"p_raw": p, "alpha_corrected": corrected_alpha, "significant": p < corrected_alpha}
            for name, p in p_values.items()}


## Cell 19: 8. PHASE V: Mediation decomposition (mediator models now include covariates, fix #2)

In [10]:
# ===== CELL 20 =====
def _encode_covariates(sub, covariates):
    """
    Build a numeric covariate matrix, one-hot encoding any categorical/object columns
    instead of coercing them with pd.to_numeric (which silently turns an entire
    categorical column, e.g. gender or a bucketed age string like '[70-80)', into NaN).
    Returns a numeric DataFrame; column count may exceed len(covariates) due to one-hot.
    """
    parts = []
    for c in covariates:
        col = sub[c]
        if pd.api.types.is_numeric_dtype(col):
            parts.append(col.rename(c).astype(float))
        else:
            dummies = pd.get_dummies(col.astype(str), prefix=c, drop_first=True).astype(float)
            for dc in dummies.columns:
                parts.append(dummies[dc])
    if parts:
        mat = pd.concat(parts, axis=1)
    else:
        mat = pd.DataFrame(index=sub.index)
    return mat.fillna(mat.mean(numeric_only=True)).fillna(0.0)


def mediation_decomposition(df, attr_col, a0, a1, mediators, outcome_col, covariates,
                             n_boot=BOOTSTRAP_B, is_pilot=False):
    """
    TV_{a0,a1}(y) = SE_{a0,a1}(y) + IE_{a0,a1}(y|a1) - DE_{a1,a0}(y|a1)
    => SE = TV - IE + DE     (fix #1: corrected comment/derivation)

    Fix #2: mediator models are now M ~ A + covariates (not just M ~ A), so the simulated
    natural-effect mediators reflect the patient's other characteristics, not only their
    protected-attribute value. This is still a parametric approximation (a full g-formula
    would model the joint mediator distribution and any mediator-mediator dependence),
    but is a materially better approximation of the natural-effects definition than a
    mediator model conditioned on A alone.

    Bug fix (post-first-run): covariates are no longer coerced with pd.to_numeric, which
    silently turned categorical columns (e.g. gender, bucketed age strings) into all-NaN
    and crashed LogisticRegression.fit(). Categorical covariates are now one-hot encoded
    via _encode_covariates() instead.
    """
    sub = df[df[attr_col].isin([a0, a1])].copy()
    sub["a_bin"] = (sub[attr_col] == a1).astype(int)
    covariates = [c for c in covariates if c in sub.columns]

    cov_df = _encode_covariates(sub, covariates)
    cov_cols = list(cov_df.columns)
    cov_matrix = cov_df.values if cov_cols else np.zeros((len(sub), 0))

    X_out = pd.concat(
        [sub[["a_bin"]].reset_index(drop=True),
         sub[mediators].apply(pd.to_numeric, errors="coerce").reset_index(drop=True),
         cov_df.reset_index(drop=True)],
        axis=1,
    )
    for m in mediators:
        X_out[f"a_x_{m}"] = X_out["a_bin"] * X_out[m]
    X_out = X_out.fillna(X_out.mean(numeric_only=True)).fillna(0.0)
    y_out = sub[outcome_col].values

    outcome_model = LogisticRegression(max_iter=1000)
    outcome_model.fit(X_out.values, y_out)

    mediator_models = {}
    for m in mediators:
        Xm = np.column_stack([sub["a_bin"].values, cov_matrix]) if cov_cols else \
            sub["a_bin"].values.reshape(-1, 1)
        ym = pd.to_numeric(sub[m], errors="coerce")
        ym = ym.fillna(ym.mean()).values
        if set(np.unique(ym)) <= {0, 1}:
            lr = LogisticRegression(max_iter=500).fit(Xm, ym)
            mediator_models[m] = ("logistic", lr)
        else:
            from sklearn.linear_model import Ridge
            reg = Ridge().fit(Xm, ym)
            mediator_models[m] = ("linear", reg)

    def simulate_outcome(a_for_pred, a_for_mediators, rng):
        n = len(sub)
        Xm_input = np.column_stack([np.full(n, a_for_mediators), cov_matrix]) if cov_cols else \
            np.full((n, 1), a_for_mediators)
        row = {"a_bin": np.full(n, a_for_pred)}
        for m in mediators:
            kind, mdl = mediator_models[m]
            if kind == "logistic":
                p = mdl.predict_proba(Xm_input)[:, 1]
                row[m] = rng.binomial(1, np.clip(p, 0, 1))
            else:
                pred = mdl.predict(Xm_input)
                row[m] = pred + rng.normal(0, 0.1, n)
        for i, c in enumerate(cov_cols):
            row[c] = cov_matrix[:, i]
        X = pd.DataFrame(row)
        for m in mediators:
            X[f"a_x_{m}"] = X["a_bin"] * X[m]
        X = X[X_out.columns]
        return outcome_model.predict_proba(X.values)[:, 1]

    def one_draw(rng):
        y_11 = simulate_outcome(1, 1, rng).mean()
        y_00 = simulate_outcome(0, 0, rng).mean()
        y_10 = simulate_outcome(1, 0, rng).mean()
        TV = y_11 - y_00
        DE = y_10 - y_00
        IE = y_11 - y_10
        SE = TV - IE + DE  # fix #1
        return TV, SE, IE, DE

    if is_pilot:
        rng = np.random.RandomState(0)
        TV, SE, IE, DE = one_draw(rng)
        return {"TV": TV, "SE": SE, "IE": IE, "DE": DE, "CI": None,
                "note": "pilot, unpowered, point estimate only"}

    rng = np.random.RandomState(0)
    acc = {"TV": [], "SE": [], "IE": [], "DE": []}
    for _ in range(n_boot):
        TV, SE, IE, DE = one_draw(rng)
        acc["TV"].append(TV); acc["SE"].append(SE); acc["IE"].append(IE); acc["DE"].append(DE)

    return {k: {"mean": float(np.mean(v)), "ci_lo": float(np.percentile(v, 2.5)),
                "ci_hi": float(np.percentile(v, 97.5))} for k, v in acc.items()}




## Cell 21: 9. LOHO

In [11]:
# ===== CELL 22 =====
def run_loho(df, feature_cols, y_col, env_col, systematic_sample_every=None, max_folds=None):
    envs = sorted(df[env_col].unique())
    if systematic_sample_every:
        envs = envs[::systematic_sample_every]
    if max_folds:
        envs = envs[:max_folds]

    results = []
    X_all = pd.get_dummies(df[feature_cols], drop_first=True)
    for env in envs:
        test_mask = df[env_col] == env
        if test_mask.sum() < 5 or (~test_mask).sum() < 5:
            continue
        X_train, X_test = X_all[~test_mask], X_all[test_mask]
        y_train, y_test = df.loc[~test_mask, y_col], df.loc[test_mask, y_col]
        if y_train.nunique() < 2 or y_test.nunique() < 2:
            continue
        clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
        probs = clf.predict_proba(X_test)[:, 1]
        results.append({"env": env, "n_test": int(test_mask.sum()), "AUROC": roc_auc_score(y_test, probs)})
    return pd.DataFrame(results)


## Cell 23: 10. CLOSED-LOOP AUDIT

In [12]:
# ===== CELL 24 =====
def closed_loop_audit(df, risk_scores, protected_group, iterations=CLOSED_LOOP_ITERS,
                       top_pct=0.20, effect_reduction=0.10):
    df = df.copy()
    df["risk"] = risk_scores
    df["y_sim"] = df["y"].astype(float)
    history = []
    for it in range(iterations):
        threshold = df["risk"].quantile(1 - top_pct)
        treated = df["risk"] >= threshold
        df.loc[treated, "y_sim"] = df.loc[treated, "y_sim"] * (1 - effect_reduction)
        preds_bin = (df["risk"] > 0.5).astype(int)
        y_bin = (df["y_sim"] > 0.5).astype(int)
        for g in np.unique(protected_group):
            mask = protected_group == g
            pos_mask = mask & (y_bin == 1)
            fnr = np.nan if pos_mask.sum() == 0 else (preds_bin[pos_mask] == 0).mean()
            history.append({"iteration": it, "group": g, "FNR": fnr})
        noise = np.random.normal(0, 0.01, size=len(df))
        df["risk"] = np.clip(df["risk"] * 0.98 + df["y_sim"] * 0.02 + noise, 0, 1)
    return pd.DataFrame(history)


## Cell 25: 11. BASELINES

In [13]:
# ===== CELL 26 =====
def run_baselines(X_train, y_train, X_test, y_test, group_test, group_train):
    results = {}
    lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    results["LogisticRegression"] = evaluate_predictions(y_test, lr.predict_proba(X_test)[:, 1], group_test)

    try:
        import xgboost as xgb
        clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, eval_metric="logloss")
        clf.fit(X_train, y_train)
        results["XGBoost"] = evaluate_predictions(y_test, clf.predict_proba(X_test)[:, 1], group_test)
    except ImportError:
        from sklearn.ensemble import GradientBoostingClassifier
        gbc = GradientBoostingClassifier().fit(X_train, y_train)
        results["GradientBoosting_fallback"] = evaluate_predictions(
            y_test, gbc.predict_proba(X_test)[:, 1], group_test)

    try:
        from fairlearn.reductions import ExponentiatedGradient, DemographicParity
        mitigator = ExponentiatedGradient(LogisticRegression(max_iter=1000), constraints=DemographicParity())
        mitigator.fit(X_train, y_train, sensitive_features=group_train)
        preds = mitigator.predict(X_test)
        results["Fairlearn_Reductions"] = evaluate_predictions(y_test, preds.astype(float), group_test)
    except Exception as e:
        results["Fairlearn_Reductions"] = {"note": f"skipped ({e})"}

    return results


## Cell 27: 12. HYPERPARAMETER SWEEP (fix #4) + EPSILON SENSITIVITY (fix #5)

In [14]:
# ===== CELL 28 =====
def sweep_hyperparameters(C_train, y_train, env_train, group_train, C_val, y_val, group_val,
                           lambda_grid=LAMBDA_INV_GRID, gamma_grid=GAMMA_FAIR_GRID,
                           epsilon=EPSILON_MAIN, quick_epochs=20):
    """Fix #4: grid search over (lambda_inv, gamma_fair), selected by validation AUROC
    subject to validation EOD <= epsilon (utility-maximizing among fair-enough configs;
    if none satisfy the constraint, falls back to the config with smallest EOD).

    Fixed after the first real run: this used to call train_predictor() TWICE per grid
    point (once discarded, once used) -- pure waste, now removed. It also now passes the
    validation set into train_predictor() itself so early stopping (see Cell 16) applies
    during the sweep too, not just at final training time.
    """
    best_cfg, best_score, best_feasible = None, -np.inf, False
    fallback_cfg, fallback_eod = None, np.inf
    records = []

    for lam_inv, gam_fair in itertools.product(lambda_grid, gamma_grid):
        model, _ = train_predictor(C_train, y_train, env_train, group_train,
                                    lambda_inv=lam_inv, gamma_fair=gam_fair, epsilon=epsilon,
                                    epochs=quick_epochs, C_val=C_val, y_val=y_val, group_val=group_val)
        with torch.no_grad():
            val_logits = model(torch.tensor(C_val, dtype=torch.float32).to(DEVICE))
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
        m = evaluate_predictions(y_val, val_probs, group_val)
        records.append({"lambda_inv": lam_inv, "gamma_fair": gam_fair, **m})

        if m["EOD"] <= epsilon and m["AUROC"] > best_score:
            best_score, best_cfg, best_feasible = m["AUROC"], (lam_inv, gam_fair), True
        if m["EOD"] < fallback_eod:
            fallback_eod, fallback_cfg = m["EOD"], (lam_inv, gam_fair)

    selected = best_cfg if best_feasible else fallback_cfg
    return selected, pd.DataFrame(records)


def epsilon_sensitivity_table(C_train, y_train, env_train, group_train, C_test, y_test, group_test,
                               lambda_inv, gamma_fair, epsilons=EPSILON_SWEEP):
    """Fix #5: Table 7 — sweep epsilon, report test-set AUROC/EOD for each."""
    rows = []
    for eps in epsilons:
        model, _ = train_predictor(C_train, y_train, env_train, group_train,
                                    lambda_inv=lambda_inv, gamma_fair=gamma_fair, epsilon=eps)
        with torch.no_grad():
            logits = model(torch.tensor(C_test, dtype=torch.float32).to(DEVICE))
            probs = torch.sigmoid(logits).cpu().numpy()
        m = evaluate_predictions(y_test, probs, group_test)
        rows.append({"epsilon": eps, **m})
    return pd.DataFrame(rows)


## Cell 29: 13. ABLATION STUDY (fix #6)

In [15]:
# ===== CELL 30 =====
def run_ablations(spec, df, feature_cols, lambda_inv, gamma_fair, lambda_adv, epsilon=EPSILON_MAIN):
    """
    Fix #6: runs the four ablations specified in experimental_design_v2.tex:
      -IRM             : lambda_inv forced to 0
      -SCD             : predictor trained directly on DCEVAE latent U, skipping SCD
      -FairImputation  : naive mean/mode imputation instead of MNAR-aware strategy
      -PCFairness      : demographic parity constraint instead of Equalized-Odds-based PC-fairness
    Returns a dict of test-set metrics per ablation, plus the full ("A-ICF (full)") config for reference.
    """
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        pd.get_dummies(df[feature_cols], drop_first=True).fillna(0).values.astype(np.float32),
        df["y"].values.astype(np.float32),
        np.arange(len(df)), test_size=0.15, random_state=0, stratify=df["y"].values
    )
    env_all = df[spec.env_col].astype("category")
    env_codes_all = env_all.cat.codes.values
    n_envs = env_all.cat.categories.size
    group_all = df[spec.primary_attr].values

    env_train, env_test = env_codes_all[idx_train], env_codes_all[idx_test]
    group_train, group_test = group_all[idx_train], group_all[idx_test]

    def encode(X):
        input_layer, backbone, dbn, _ = train_dcevae(X, apply_dbn=spec.temporal)
        with torch.no_grad():
            h = input_layer(torch.tensor(X, dtype=torch.float32).to(DEVICE))
            _, u, _, _ = backbone(h)
        return u.cpu().numpy()

    U_train, U_test = encode(X_train), encode(X_test)

    results = {}

    # Full A-ICF
    _, C_train, _ = train_scd(U_train, env_train, n_envs, lambd_adv=lambda_adv, epochs=10)
    model_full, _ = train_predictor(C_train, y_train, env_train, group_train,
                                     lambda_inv=lambda_inv, gamma_fair=gamma_fair, epsilon=epsilon)
    _, C_test, _ = train_scd(U_test, env_test, n_envs, lambd_adv=lambda_adv, epochs=10)
    with torch.no_grad():
        probs_full = torch.sigmoid(model_full(torch.tensor(C_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    results["A-ICF (full)"] = evaluate_predictions(y_test, probs_full, group_test)

    # -IRM
    model_noirm, _ = train_predictor(C_train, y_train, env_train, group_train,
                                      lambda_inv=lambda_inv, gamma_fair=gamma_fair, epsilon=epsilon,
                                      use_irm=False)
    with torch.no_grad():
        probs_noirm = torch.sigmoid(model_noirm(torch.tensor(C_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    results["-IRM"] = evaluate_predictions(y_test, probs_noirm, group_test)

    # -SCD (predictor on raw U instead of SCD content C)
    model_nosdc, _ = train_predictor(U_train, y_train, env_train, group_train,
                                      lambda_inv=lambda_inv, gamma_fair=gamma_fair, epsilon=epsilon)
    with torch.no_grad():
        probs_nosdc = torch.sigmoid(model_nosdc(torch.tensor(U_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    results["-SCD"] = evaluate_predictions(y_test, probs_nosdc, group_test)

    # -PCFairness (demographic parity instead of EOD-based path-specific fairness)
    model_dp, _ = train_predictor(C_train, y_train, env_train, group_train,
                                   lambda_inv=lambda_inv, gamma_fair=gamma_fair, epsilon=epsilon,
                                   use_pc_fairness=False)
    with torch.no_grad():
        probs_dp = torch.sigmoid(model_dp(torch.tensor(C_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    results["-PCFairness"] = evaluate_predictions(y_test, probs_dp, group_test)

    # -FairImputation: reload data with naive imputation, redo the full pipeline quickly
    if spec.name == "diabetic_data":
        df_naive = load_diabetic_data(fair_imputation=False)
    elif spec.name == "framingham":
        df_naive = load_framingham(fair_imputation=False)
    else:
        df_naive = df  # MIMIC-IV has no MICE step to ablate (MNAR indicator retained either way)

    X_naive = pd.get_dummies(df_naive[feature_cols], drop_first=True).fillna(0).values.astype(np.float32)
    X_naive_train, X_naive_test = X_naive[idx_train], X_naive[idx_test]
    U_naive_train = encode(X_naive_train)
    U_naive_test = encode(X_naive_test)
    _, C_naive_train, _ = train_scd(U_naive_train, env_train, n_envs, lambd_adv=lambda_adv, epochs=10)
    _, C_naive_test, _ = train_scd(U_naive_test, env_test, n_envs, lambd_adv=lambda_adv, epochs=10)
    model_naive, _ = train_predictor(C_naive_train, y_train, env_train, group_train,
                                      lambda_inv=lambda_inv, gamma_fair=gamma_fair, epsilon=epsilon)
    with torch.no_grad():
        probs_naive = torch.sigmoid(
            model_naive(torch.tensor(C_naive_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    results["-FairImputation"] = evaluate_predictions(y_test, probs_naive, group_test)

    return results


## Cell 31: 14. ORCHESTRATION (with proper train/val/test split, fix #9)

In [16]:
# ===== CELL 32 =====
def run_dataset_pipeline(spec: DatasetSpec, df: pd.DataFrame, feature_cols: list):
    print(f"\n{'='*70}\n{spec.name} (pilot={spec.is_pilot})\n{'='*70}")

    X_native = pd.get_dummies(df[feature_cols], drop_first=True).fillna(0).values.astype(np.float32)
    y = df["y"].values.astype(np.float32)
    env_cat = df[spec.env_col].astype("category")
    env_codes = env_cat.cat.codes.values
    n_envs = env_cat.cat.categories.size
    group = df[spec.primary_attr].values
    batch_size = BATCH_SIZE_SMALL if spec.is_pilot else BATCH_SIZE_LARGE

    # Fix #9: proper 70/15/15 split for the large dataset; 5-fold CV noted for small ones
    # (here implemented as a single held-out 70/15/15 split per fold-equivalent for simplicity;
    # wrap this function in a KFold loop externally to get the full 5-fold estimate).
    idx = np.arange(len(df))
    idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=0, stratify=y)
    idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=0, stratify=y[idx_temp])

    def subset(arr):
        return arr[idx_train], arr[idx_val], arr[idx_test]

    X_train, X_val, X_test = subset(X_native)
    y_train, y_val, y_test = subset(y)
    env_train, env_val, env_test = subset(env_codes)
    group_train, group_val, group_test = subset(group)

    if spec.is_pilot:
        print(f"[PILOT WARNING] {spec.name}: N={len(df)} — running a single fixed configuration, "
              "NOT the full hyperparameter/epsilon sweep (a 3x3 grid would overfit validation "
              "signal at this sample size). Results are descriptive only.")

    # Phase II — trained on TRAIN split only
    input_layer, backbone, dbn, elbo = train_dcevae(X_train, batch_size=batch_size, apply_dbn=spec.temporal)

    def get_latent(X):
        with torch.no_grad():
            h = input_layer(torch.tensor(X, dtype=torch.float32).to(DEVICE))
            _, u, _, _ = backbone(h)
        return u.cpu().numpy()

    U_train, U_val, U_test = get_latent(X_train), get_latent(X_val), get_latent(X_test)

    # Phase III — lambda_adv selection (fix #14), skipped (fixed value) for pilot
    if spec.is_pilot:
        lambda_adv_selected = 0.1
        scd_model, C_train, site_acc = train_scd(U_train, env_train, n_envs, lambd_adv=lambda_adv_selected)
    else:
        lambda_adv_selected, scd_model, C_train, chance_acc = select_lambda_adv(U_train, env_train, n_envs)
        _, _, site_acc = train_scd(U_train, env_train, n_envs, lambd_adv=lambda_adv_selected)

    def apply_scd(U):
        with torch.no_grad():
            c, _, _, _ = scd_model(torch.tensor(U, dtype=torch.float32).to(DEVICE), lambda_adv_selected)
        return c.cpu().numpy()

    C_val, C_test = apply_scd(U_val), apply_scd(U_test)

    # Phase IV — hyperparameter sweep (fix #4) + epsilon sensitivity (fix #5), skipped for pilot
    if spec.is_pilot:
        selected_hparams = (1.0, 1.0)
        sweep_records = pd.DataFrame([{"note": "sweep skipped for pilot dataset"}])
        eps_table = pd.DataFrame([{"note": "epsilon sweep skipped for pilot dataset"}])
    else:
        selected_hparams, sweep_records = sweep_hyperparameters(
            C_train, y_train, env_train, group_train, C_val, y_val, group_val
        )
        eps_table = epsilon_sensitivity_table(
            C_train, y_train, env_train, group_train, C_test, y_test, group_test,
            lambda_inv=selected_hparams[0], gamma_fair=selected_hparams[1]
        )

    lambda_inv_sel, gamma_fair_sel = selected_hparams
    # Validation set now passed through for early stopping (Cell 16) on the final model too,
    # not just during the sweep -- this was the main structural fix after AUROC collapsed
    # to ~random across the whole grid on the first real run.
    final_model, _ = train_predictor(C_train, y_train, env_train, group_train,
                                      lambda_inv=lambda_inv_sel, gamma_fair=gamma_fair_sel,
                                      epsilon=EPSILON_MAIN,
                                      C_val=C_val, y_val=y_val, group_val=group_val)
    with torch.no_grad():
        test_probs = torch.sigmoid(final_model(torch.tensor(C_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    test_metrics = evaluate_predictions(y_test, test_probs, group_test)

    # Baselines on the same split
    baseline_metrics = run_baselines(X_train, y_train, X_test, y_test, group_test, group_train)

    # Bonferroni-corrected significance vs. baselines (fix #10), using 5-seed AUROC arrays
    seed_aucs_aicf = []
    for s in SEEDS:
        torch.manual_seed(s)
        m, _ = train_predictor(C_train, y_train, env_train, group_train,
                                lambda_inv=lambda_inv_sel, gamma_fair=gamma_fair_sel, epsilon=EPSILON_MAIN)
        with torch.no_grad():
            p = torch.sigmoid(m(torch.tensor(C_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
        seed_aucs_aicf.append(roc_auc_score(y_test, p))

    p_values = {}
    lr_baseline = LogisticRegression(max_iter=1000)
    for name in baseline_metrics:
        if isinstance(baseline_metrics[name], dict) and "AUROC" in baseline_metrics[name]:
            baseline_auc_repeat = [baseline_metrics[name]["AUROC"]] * len(SEEDS)  # baseline is deterministic here
            p_values[name] = paired_bootstrap_pvalue(seed_aucs_aicf, baseline_auc_repeat)
    significance = bonferroni_correct(p_values, n_comparisons=max(len(p_values), 1))

    # E-value + mediation
    a0, a1 = spec.primary_contrast
    rr = compute_risk_ratio(df, spec.primary_attr, "y", a0, a1)
    ev = e_value(rr) if not np.isnan(rr) else np.nan
    covariate_cols = [c for c in feature_cols if c not in spec.mediators][:5]  # small covariate set
    mediation = mediation_decomposition(df, spec.primary_attr, a0, a1, spec.mediators, "y",
                                         covariates=covariate_cols, is_pilot=spec.is_pilot)

    return {
        "dataset": spec.name,
        "is_pilot": spec.is_pilot,
        "elbo": elbo,
        "lambda_adv_selected": lambda_adv_selected,
        "site_classifier_acc": site_acc,
        "selected_hyperparams": {"lambda_inv": lambda_inv_sel, "gamma_fair": gamma_fair_sel},
        "hyperparam_sweep": sweep_records.to_dict(orient="records"),
        "epsilon_sensitivity": eps_table.to_dict(orient="records"),
        "test_metrics": test_metrics,
        "baseline_metrics": baseline_metrics,
        "significance_vs_baselines": significance,
        "risk_ratio": rr,
        "e_value": ev,
        "mediation": mediation,
    }, test_probs, C_test, idx_test


## Cell 33: 16. FIGURE GENERATION FUNCTIONS (matplotlib, saved as PNG)

In [17]:
# ===== CELL 34 =====
# NOTE: added so the pipeline produces actual image files, not just JSON/CSV. Each function
# writes a PNG to OUT_DIR/figures/ and returns its path. Figures are matched to the Table
# 4-7 / Figure 3-7 deliverables listed in experimental_design_v2.tex.

def plot_fairness_utility_tradeoff(sweep_data, dataset_name, output_dir):
    """Figure 3: Fairness-utility trade-off curve, AUROC vs EOD across swept configs."""
    fig, ax = plt.subplots(figsize=(8, 6))
    df = pd.DataFrame(sweep_data)
    scatter = ax.scatter(df["EOD"], df["AUROC"], c=df["gamma_fair"], cmap="viridis", s=100, alpha=0.7)
    ax.set_xlabel("EOD (Fairness Violation)", fontsize=12)
    ax.set_ylabel("AUROC", fontsize=12)
    ax.set_title(f"Fairness-Utility Trade-off: {dataset_name}", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Random (0.5)")
    ax.legend()
    cbar = plt.colorbar(scatter)
    cbar.set_label("gamma_fair")
    plt.tight_layout()
    fig_path = os.path.join(output_dir, "figures", f"figure3_tradeoff_{dataset_name}.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    return fig_path


def plot_loho_boxplot(loho_df, output_dir):
    """Figure 4: LOHO cross-site AUC stability box plot."""
    fig, ax = plt.subplots(figsize=(10, 6))
    valid_df = loho_df[loho_df["n_test"] >= 30] if "n_test" in loho_df.columns else loho_df
    if valid_df.empty or "AUROC" not in valid_df.columns:
        plt.close()
        return None
    ax.boxplot(valid_df["AUROC"].dropna())
    ax.set_ylabel("AUROC", fontsize=12)
    ax.set_title("Leave-One-Hospital-Out Cross-Site Stability", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Random (0.5)")
    ax.legend()
    plt.tight_layout()
    fig_path = os.path.join(output_dir, "figures", "figure4_loho_boxplot.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    return fig_path


def plot_bias_decomposition(mediation_results, output_dir):
    """Figure 5: Bias decomposition bar chart (SE, IE, DE, TV). Pilot datasets (no CI) skipped."""
    fig, ax = plt.subplots(figsize=(10, 6))
    datasets, se_values, ie_values, de_values, tv_values = [], [], [], [], []
    for name, res in mediation_results.items():
        if res.get("CI") is None and "TV" not in res:
            continue
        if isinstance(res.get("TV"), dict):  # bootstrapped (non-pilot) result
            datasets.append(name.replace("_", " ").title())
            tv_values.append(res["TV"]["mean"]); se_values.append(res["SE"]["mean"])
            ie_values.append(res["IE"]["mean"]); de_values.append(res["DE"]["mean"])
    if not datasets:
        plt.close()
        return None
    x = np.arange(len(datasets))
    width = 0.2
    ax.bar(x - 1.5*width, se_values, width, label="SE (Spurious)", color="#ff9999", edgecolor="black")
    ax.bar(x - 0.5*width, ie_values, width, label="IE (Indirect)", color="#66b3ff", edgecolor="black")
    ax.bar(x + 0.5*width, de_values, width, label="DE (Direct)", color="#99ff99", edgecolor="black")
    ax.bar(x + 1.5*width, tv_values, width, label="TV (Total)", color="#ffcc99", edgecolor="black")
    ax.set_xlabel("Dataset", fontsize=12)
    ax.set_ylabel("Effect Size", fontsize=12)
    ax.set_title("Bias Decomposition: SE, IE, DE, TV", fontsize=14)
    ax.set_xticks(x); ax.set_xticklabels(datasets)
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color="black", linestyle="-", alpha=0.5)
    plt.tight_layout()
    fig_path = os.path.join(output_dir, "figures", "figure5_bias_decomposition.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    return fig_path


def plot_closed_loop_audit(audit_df, output_dir):
    """Figure 6: Closed-loop audit -- subgroup FNR drift over iterations."""
    fig, ax = plt.subplots(figsize=(10, 6))
    clean_df = audit_df.dropna(subset=["FNR"])
    for group in clean_df["group"].unique():
        subset = clean_df[clean_df["group"] == group]
        ax.plot(subset["iteration"], subset["FNR"], marker="o", linewidth=2, label=group)
    ax.set_xlabel("Iteration", fontsize=12)
    ax.set_ylabel("False Negative Rate", fontsize=12)
    ax.set_title("Closed-Loop Audit: Subgroup FNR Drift", fontsize=14)
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    fig_path = os.path.join(output_dir, "figures", "figure6_closed_loop_audit.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    return fig_path


def plot_ablation_comparison(ablation_results, dataset_name, output_dir):
    """Figure 7: Ablation study comparison bar chart (AUROC + EOD per configuration)."""
    fig, ax = plt.subplots(figsize=(10, 6))
    names = list(ablation_results.keys())
    auroc_values = [ablation_results[n].get("AUROC", 0) for n in names]
    eod_values = [ablation_results[n].get("EOD", 0) for n in names]
    x = np.arange(len(names))
    width = 0.35
    bars1 = ax.bar(x - width/2, auroc_values, width, label="AUROC", color="#3498db", edgecolor="black")
    bars2 = ax.bar(x + width/2, eod_values, width, label="EOD", color="#e74c3c", edgecolor="black")
    ax.set_xlabel("Ablation Configuration", fontsize=12)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"Ablation Study: {dataset_name}", fontsize=14)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=45, ha="right")
    ax.legend(); ax.grid(True, alpha=0.3)
    for i, name in enumerate(names):
        if name == "A-ICF (full)":
            bars1[i].set_color("#2ecc71")
            bars2[i].set_color("#2ecc71")
    plt.tight_layout()
    fig_path = os.path.join(output_dir, "figures", f"figure7_ablation_{dataset_name}.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    return fig_path


## Cell 35: 17. TABLE GENERATION FUNCTIONS (LaTeX)

In [18]:
# ===== CELL 36 =====
# NOTE: added to emit publication-ready LaTeX tables directly from all_results, matching
# Tables 4-7 in experimental_design_v2.tex. Pilot datasets (MIMIC-IV) are excluded from
# these tables since their results are descriptive-only, not statistically powered.

def generate_table_main_results(all_results, output_dir):
    """Table 4: Main results (AUROC, F1, Brier, EOD, DPD)."""
    rows = []
    for dataset, results in all_results.items():
        if results.get("is_pilot", False):
            continue
        metrics = results.get("test_metrics", {})
        rows.append({
            "Dataset": dataset.replace("_", " ").title(),
            "AUROC": f"{metrics.get('AUROC', 0):.3f}",
            "F1": f"{metrics.get('F1', 0):.3f}",
            "Brier": f"{metrics.get('Brier', 0):.4f}",
            "EOD": f"{metrics.get('EOD', 0):.3f}",
            "DPD": f"{metrics.get('DPD', 0):.3f}",
        })
    if not rows:
        return
    with open(os.path.join(output_dir, "tables", "table4_main_results.tex"), "w") as f:
        f.write("%% Table 4: Main Results\n")
        f.write("\\begin{table}[H]\n\\centering\n")
        f.write("\\caption{Main Results: Predictive Performance and Fairness Metrics}\n")
        f.write("\\label{tab:main_results}\n")
        f.write("\\begin{tabular}{lccccc}\n\\toprule\n")
        f.write("Dataset & AUROC & F1 & Brier & EOD & DPD \\\\\n\\midrule\n")
        for row in rows:
            f.write(f"{row['Dataset']} & {row['AUROC']} & {row['F1']} & {row['Brier']} & "
                    f"{row['EOD']} & {row['DPD']} \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")


def generate_table_baseline_comparison(all_results, output_dir):
    """Table 4 (extended): Baseline comparison, AUROC only."""
    rows = []
    for dataset, results in all_results.items():
        if results.get("is_pilot", False):
            continue
        baselines = results.get("baseline_metrics", {})
        test_metrics = results.get("test_metrics", {})
        row = {"Dataset": dataset.replace("_", " ").title(), "A-ICF": f"{test_metrics.get('AUROC', 0):.3f}"}
        for name, metrics in baselines.items():
            if isinstance(metrics, dict) and "AUROC" in metrics:
                row[name] = f"{metrics['AUROC']:.3f}"
        rows.append(row)
    if not rows:
        return
    with open(os.path.join(output_dir, "tables", "table4_baseline_comparison.tex"), "w") as f:
        f.write("%% Table 4 (Extended): Baseline Comparison\n")
        f.write("\\begin{table}[H]\n\\centering\n")
        f.write("\\caption{Baseline Comparison (AUROC)}\n\\label{tab:baseline_comparison}\n")
        cols = [k for k in rows[0].keys() if k != "Dataset"]
        f.write("\\begin{tabular}{l" + "c" * len(cols) + "}\n\\toprule\n")
        f.write("Dataset & " + " & ".join(cols) + " \\\\\n\\midrule\n")
        for row in rows:
            f.write(f"{row['Dataset']} & " + " & ".join(row[k] for k in cols) + " \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")


def generate_table_bias_decomposition(mediation_results, output_dir):
    """Table 5: Bias decomposition (SE, IE, DE with 95% CI). Pilot (no CI) results excluded."""
    rows = []
    for dataset, res in mediation_results.items():
        if not isinstance(res.get("TV"), dict):
            continue
        rows.append({
            "Dataset": dataset.replace("_", " ").title(),
            "TV": f"{res['TV']['mean']:.4f} [{res['TV']['ci_lo']:.4f}, {res['TV']['ci_hi']:.4f}]",
            "SE": f"{res['SE']['mean']:.4f} [{res['SE']['ci_lo']:.4f}, {res['SE']['ci_hi']:.4f}]",
            "IE": f"{res['IE']['mean']:.4f} [{res['IE']['ci_lo']:.4f}, {res['IE']['ci_hi']:.4f}]",
            "DE": f"{res['DE']['mean']:.4f} [{res['DE']['ci_lo']:.4f}, {res['DE']['ci_hi']:.4f}]",
        })
    if not rows:
        return
    with open(os.path.join(output_dir, "tables", "table5_bias_decomposition.tex"), "w") as f:
        f.write("%% Table 5: Bias Decomposition\n")
        f.write("\\begin{table}[H]\n\\centering\n")
        f.write("\\caption{Bias Decomposition: $TV = SE + IE - DE$ with 95\\%% CI}\n")
        f.write("\\label{tab:bias_decomposition}\n\\begin{tabular}{lcccc}\n\\toprule\n")
        f.write("Dataset & TV & SE & IE & DE \\\\\n\\midrule\n")
        for row in rows:
            f.write(f"{row['Dataset']} & {row['TV']} & {row['SE']} & {row['IE']} & {row['DE']} \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")


def generate_table_ablation(ablation_results, output_dir):
    """Table 6: Ablation study."""
    rows = []
    for dataset, ablations in ablation_results.items():
        for name, metrics in ablations.items():
            rows.append({
                "Dataset": dataset.replace("_", " ").title(),
                "Configuration": name,
                "AUROC": f"{metrics.get('AUROC', 0):.3f}",
                "F1": f"{metrics.get('F1', 0):.3f}",
                "EOD": f"{metrics.get('EOD', 0):.3f}",
                "DPD": f"{metrics.get('DPD', 0):.3f}",
            })
    if not rows:
        return
    with open(os.path.join(output_dir, "tables", "table6_ablation.tex"), "w") as f:
        f.write("%% Table 6: Ablation Study\n")
        f.write("\\begin{table}[H]\n\\centering\n")
        f.write("\\caption{Ablation Study}\n\\label{tab:ablation}\n")
        f.write("\\begin{tabular}{lccccc}\n\\toprule\n")
        f.write("Dataset & Configuration & AUROC & F1 & EOD & DPD \\\\\n\\midrule\n")
        for row in rows:
            f.write(f"{row['Dataset']} & {row['Configuration']} & {row['AUROC']} & {row['F1']} & "
                    f"{row['EOD']} & {row['DPD']} \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")


def generate_table_epsilon_sensitivity(all_results, output_dir):
    """Table 7: Epsilon-sensitivity sweep."""
    rows = []
    for dataset, results in all_results.items():
        if results.get("is_pilot", False):
            continue
        for record in results.get("epsilon_sensitivity", []):
            if "note" in record:
                continue
            rows.append({
                "Dataset": dataset.replace("_", " ").title(),
                "Epsilon": record.get("epsilon", 0),
                "AUROC": f"{record.get('AUROC', 0):.3f}",
                "EOD": f"{record.get('EOD', 0):.3f}",
                "DPD": f"{record.get('DPD', 0):.3f}",
            })
    if not rows:
        return
    with open(os.path.join(output_dir, "tables", "table7_epsilon_sensitivity.tex"), "w") as f:
        f.write("%% Table 7: Epsilon Sensitivity\n")
        f.write("\\begin{table}[H]\n\\centering\n")
        f.write("\\caption{Epsilon-Sensitivity Sweep}\n\\label{tab:epsilon_sensitivity}\n")
        f.write("\\begin{tabular}{lcccc}\n\\toprule\n")
        f.write("Dataset & $\\epsilon$ & AUROC & EOD & DPD \\\\\n\\midrule\n")
        for row in rows:
            f.write(f"{row['Dataset']} & {row['Epsilon']:.2f} & {row['AUROC']} & {row['EOD']} & "
                    f"{row['DPD']} \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")


## Cell 37: 18. MAIN

In [19]:
# ===== CELL 38 =====
def main():
    print("="*70)
    print("A-ICF v2: Publication-Ready Experiment")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)

    print("\n[Phase I] Loading and preprocessing data...")
    diabetic = load_diabetic_data()
    framingham = load_framingham()
    mimic = load_mimic()

    print("\n[Phase II] Validating causal discovery module on synthetic data (enforced check)...")
    synth_check = run_synthetic_discovery_check()
    print("  Synthetic SHD check:", synth_check)

    # NOTE: mimic_iv is kept in this run (unlike an earlier draft of this cell that dropped
    # it silently). Its pilot status already excludes it from Tables 4-7 and the bootstrap-CI
    # figures via the is_pilot / "CI is None" checks throughout -- dropping the dataset
    # entirely would be a separate, bigger decision than a code cleanup, and isn't made here.
    # ALSO NOTE: MIMIC-IV's baseline AUROC/F1 were both 1.0 in the last run -- a near-certain
    # data leakage signal (e.g. discharge_location containing a "died"-equivalent category
    # that trivially predicts hospital_expire_flag). Investigate feature_cols["mimic_iv"]
    # for leakage before trusting or reporting any MIMIC-IV number, pilot or not.
    specs = {
        "diabetic_data": DatasetSpec("diabetic_data", False, "race", ("Caucasian", "AfricanAmerican"),
                                      ["number_inpatient", "number_emergency", "num_medications"],
                                      "y", "env"),
        "framingham": DatasetSpec("framingham", False, "primary_attr", (0, 1),
                                   ["BMI", "totChol", "sysBP", "glucose"], "y", "env", temporal=True),
        "mimic_iv": DatasetSpec("mimic_iv", True, "race", ("WHITE", "BLACK/AFRICAN AMERICAN"),
                                 ["icu_los_total", "had_icu_stay"], "y", "env"),
    }
    feature_cols = {
        "diabetic_data": ["gender", "age", "admission_type_id", "discharge_disposition_id",
                          "time_in_hospital", "num_lab_procedures", "num_procedures",
                          "num_medications", "number_outpatient", "number_emergency",
                          "number_inpatient", "number_diagnoses", "max_glu_serum", "A1Cresult",
                          "insulin", "change", "diabetesMed"],
        "framingham": ["age", "education", "currentSmoker", "cigsPerDay", "BPMeds",
                       "prevalentStroke", "prevalentHyp", "diabetes", "totChol", "sysBP",
                       "diaBP", "BMI", "heartRate", "glucose"],
        "mimic_iv": ["gender", "anchor_age", "marital_status", "insurance", "admission_type",
                     "discharge_location", "primary_diagnosis", "had_icu_stay"],
    }
    dfs = {"diabetic_data": diabetic, "framingham": framingham, "mimic_iv": mimic}

    all_results = {}
    sweep_data = []
    mediation_results = {}
    ablation_results = {}

    for name, spec in specs.items():
        print(f"\n{'='*70}\nRunning pipeline for: {name}\n{'='*70}")
        result, probs, C_test, idx_test = run_dataset_pipeline(spec, dfs[name], feature_cols[name])
        all_results[name] = result

        for record in result.get("hyperparam_sweep", []):
            if "note" not in record:
                sweep_data.append({"dataset": name, **record})

        if "mediation" in result:
            mediation_results[name] = result["mediation"]

        print(f"\nRunning ablations for {name}...")
        ablations = run_ablations(spec, dfs[name], feature_cols[name],
                                   lambda_inv=result["selected_hyperparams"]["lambda_inv"],
                                   gamma_fair=result["selected_hyperparams"]["gamma_fair"],
                                   lambda_adv=result["lambda_adv_selected"])
        all_results[name]["ablations"] = ablations
        ablation_results[name] = ablations

    print("\nRunning LOHO on diabetic_data (Phase III)...")
    loho_df = run_loho(diabetic, feature_cols["diabetic_data"], "y", "env")
    loho_df.to_csv(os.path.join(OUT_DIR, "table4_loho_results.csv"), index=False)

    print("Running closed-loop audit (Phase V) on diabetic_data...")
    _, probs_diab, _, idx_test_diab = run_dataset_pipeline(specs["diabetic_data"], diabetic,
                                                            feature_cols["diabetic_data"])
    audit_df = closed_loop_audit(diabetic.iloc[idx_test_diab].reset_index(drop=True),
                                  probs_diab, diabetic.iloc[idx_test_diab]["race"].values)
    audit_df.to_csv(os.path.join(OUT_DIR, "figure6_closed_loop_audit.csv"), index=False)

    print("\n[Figures] Generating figures...")
    if sweep_data:
        for dataset in {d["dataset"] for d in sweep_data}:
            ds_data = [d for d in sweep_data if d["dataset"] == dataset]
            plot_fairness_utility_tradeoff(ds_data, dataset, OUT_DIR)
    plot_loho_boxplot(loho_df, OUT_DIR)
    if mediation_results:
        plot_bias_decomposition(mediation_results, OUT_DIR)
    plot_closed_loop_audit(audit_df, OUT_DIR)
    for dataset, ablations in ablation_results.items():
        if ablations:
            plot_ablation_comparison(ablations, dataset, OUT_DIR)

    print("[Tables] Generating LaTeX tables...")
    generate_table_main_results(all_results, OUT_DIR)
    generate_table_baseline_comparison(all_results, OUT_DIR)
    generate_table_bias_decomposition(mediation_results, OUT_DIR)
    generate_table_ablation(ablation_results, OUT_DIR)
    generate_table_epsilon_sensitivity(all_results, OUT_DIR)

    with open(os.path.join(OUT_DIR, "all_results.json"), "w") as f:
        json.dump(all_results, f, indent=2, default=str)

    print("\n" + "="*70)
    print("EXPERIMENT COMPLETED")
    print("="*70)
    print(f"\nOutput directory: {OUT_DIR}/")
    print("Figures: ", os.path.join(OUT_DIR, "figures"))
    print("Tables (LaTeX): ", os.path.join(OUT_DIR, "tables"))
    print("all_results.json, table4_loho_results.csv, figure6_closed_loop_audit.csv")


if __name__ == "__main__":
    main()


A-ICF v2: Publication-Ready Experiment
Started: 2026-07-26 08:19:33

[Phase I] Loading and preprocessing data...

[Phase II] Validating causal discovery module on synthetic data (enforced check)...
  Synthetic SHD check: {'mean_shd': 0.55, 'std_shd': 1.3219304066402289, 'threshold_passed': True}

Running pipeline for: diabetic_data

diabetic_data (pilot=False)

Running ablations for diabetic_data...

Running pipeline for: framingham

framingham (pilot=False)

Running ablations for framingham...

Running pipeline for: mimic_iv

mimic_iv (pilot=True)
[PILOT WARNING] mimic_iv: N=275 — running a single fixed configuration, NOT the full hyperparameter/epsilon sweep (a 3x3 grid would overfit validation signal at this sample size). Results are descriptive only.

Running ablations for mimic_iv...

Running LOHO on diabetic_data (Phase III)...
Running closed-loop audit (Phase V) on diabetic_data...

diabetic_data (pilot=False)

[Figures] Generating figures...
[Tables] Generating LaTeX tables...
